# Stage 8.8 -- Inferno Feature Pipeline

Validate consolidated Mirage + Inferno Gold outputs after the scoped Inferno feature run.

In [ ]:
from pathlib import Path
import math
import matplotlib.pyplot as plt
import pandas as pd

GOLD = Path('../data/gold')
VALIDATION = GOLD / 'validation' / 'multi_map_gold'

def load(rel):
    return pd.read_parquet(GOLD / rel)

def by_map(df):
    return df['map_name'].value_counts(dropna=False).rename_axis('map_name').reset_index(name='rows') if 'map_name' in df.columns else pd.DataFrame({'rows': [len(df)]})

## Consolidated Gold Inventory

In [ ]:
datasets = {
    'round_features_mvp': 'round_features/round_features_mvp.parquet',
    'region_presence_by_round': 'region_presence/region_presence_by_round.parquet',
    'round_state_resolved': 'round_state/round_state_resolved.parquet',
    'round_features_t_side_all': 'round_features/round_features_t_side_all.parquet',
    'round_features_t_side_planted': 'round_features/round_features_t_side_planted.parquet',
    'round_features_ct_side': 'round_features/round_features_ct_side.parquet',
    'round_region_timeline': 'round_progression/round_region_timeline.parquet',
    'death_context_by_round': 'round_progression/death_context_by_round.parquet',
    'bomb_carrier_timeline': 'round_progression/bomb_carrier_timeline.parquet',
    'round_outcome_context': 'round_progression/round_outcome_context.parquet',
}
rows = []
for name, rel in datasets.items():
    df = load(rel)
    counts = df['map_name'].value_counts(dropna=False).to_dict() if 'map_name' in df.columns else {}
    rows.append({'dataset': name, 'rows_total': len(df), 'mirage_rows': counts.get('Mirage', 0), 'inferno_rows': counts.get('Inferno', 0)})
inventory = pd.DataFrame(rows)
display(inventory)

## Inferno Round State

In [ ]:
state = load('round_state/round_state_resolved.parquet')
inferno_state = state[state['map_name'].eq('Inferno')].copy()
display(inferno_state['target_team_side'].value_counts(dropna=False))
display(inferno_state['target_site_model_label'].value_counts(dropna=False))
display(inferno_state[['round_id','round_num','target_team_side','bomb_planted','planting_team','target_site_model_label','label_confidence']].head())

## Semantic And Candidate Materialization

In [ ]:
semantic = pd.read_parquet(VALIDATION / 'inferno_semantic_feature_sanity.parquet')
candidate = pd.read_parquet(VALIDATION / 'inferno_candidate_feature_materialization.parquet')
display(semantic)
display(candidate[['candidate_id','feature_name','present_on_inferno','generation_scope','coordinate_dependency','cross_map_comparison_mode','status']])

## second_mid_upper Coordinate Review

In [ ]:
proposal = pd.read_parquet(GOLD / 'maps' / 'inferno' / 'region_mapping' / 'inferno_region_mapping_proposal.parquet')
sample = pd.read_parquet(GOLD / 'maps' / 'area_discovery' / 'map_place_coordinate_sample.parquet')
second = proposal[proposal['proposed_region_id'].eq('second_mid_upper')].copy()
second_sample = sample[sample['raw_place'].isin(second['raw_place'])].copy()
display(second[['raw_place','mapping_confidence','review_status','review_basis','x_median','y_median','z_median']])
pairs = []
centers = second[['raw_place','x_median','y_median','z_median']].dropna().reset_index(drop=True)
for i, left in centers.iterrows():
    for j, right in centers.iterrows():
        if j <= i:
            continue
        pairs.append({'left': left['raw_place'], 'right': right['raw_place'], 'distance': math.dist((left['x_median'], left['y_median'], left['z_median']), (right['x_median'], right['y_median'], right['z_median']))})
pairwise = pd.DataFrame(pairs)
display(pairwise)
fig, ax = plt.subplots(figsize=(7, 6))
for place, group in second_sample.groupby('raw_place'):
    ax.scatter(group['X'], group['Y'], s=8, alpha=0.3, label=place)
ax.scatter(second['x_median'], second['y_median'], marker='x', s=80, color='black')
for _, row in second.iterrows():
    ax.text(row['x_median'], row['y_median'], row['raw_place'], fontsize=8)
ax.set_aspect('equal', adjustable='datalim')
ax.legend(loc='best')
plt.tight_layout()
print('max_center_spread=', 0 if pairwise.empty else pairwise['distance'].max())
print('vertical_spread=', second['z_median'].max() - second['z_median'].min())

## Gate Outputs

In [ ]:
for name in ['gold_scope_inventory','gold_key_collision_audit','inferno_side_dataset_summary','multi_map_gold_audit']:
    print(name)
    display(pd.read_parquet(VALIDATION / f'{name}.parquet'))